In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# ----------------------------------
# 1. CREATE DATASET (with numeric)
# ----------------------------------
n = 20

data = pd.DataFrame({
    "Outlook": np.random.choice(["Sunny", "Rain", "Overcast"], n),
    "Temperature": np.random.choice(["Hot", "Mild", "Cool"], n),
    "Humidity": np.random.choice(["High", "Normal"], n),
    "Wind": np.random.choice(["Weak", "Strong"], n),
    "Age": np.random.randint(18, 60, n),  # Numerical attribute
    "Play": np.random.choice(["Yes", "No"], n)
})

print("Dataset:\n", data)


Dataset:
      Outlook Temperature Humidity    Wind  Age Play
0   Overcast         Hot     High  Strong   46  Yes
1      Sunny         Hot     High    Weak   35  Yes
2   Overcast        Mild     High  Strong   43  Yes
3   Overcast        Mild     High    Weak   51  Yes
4      Sunny         Hot     High  Strong   27  Yes
5      Sunny         Hot     High    Weak   53  Yes
6   Overcast         Hot   Normal    Weak   31  Yes
7       Rain        Cool   Normal  Strong   48   No
8   Overcast        Cool     High    Weak   32  Yes
9   Overcast        Cool   Normal  Strong   25   No
10  Overcast        Mild   Normal  Strong   31   No
11  Overcast        Cool   Normal  Strong   40   No
12     Sunny        Mild   Normal  Strong   57  Yes
13  Overcast        Mild     High  Strong   38  Yes
14      Rain        Cool   Normal  Strong   33  Yes
15     Sunny        Mild     High  Strong   35  Yes
16      Rain        Cool   Normal  Strong   41   No
17      Rain        Cool   Normal  Strong   43  Yes
18

In [ ]:
def entropy(y):
  probs = y.value_counts()/len(y)
  return -np.sum(probs*np.log2(probs+1e-9))
def  gini(y):
  probs = y.value_counts()/len(y)
  return 1 - np.sum(probs**2)

In [ ]:
def get_weighted_score(subsets,target,method):
  total_size = sum(len(s) for s in subsets)
  weighted = 0
  for s in subsets:
    weight = len(s)/total_size
    score = gini(s[target]) if method == "cart" else entropy(s[target])
    weighted += weight*score
  return weighted

In [ ]:
def find_best_split(data, method, target):
  best_feat,best_val,best_score = None,None,None

  for col in data.columns.drop(target):
    if np.issubdtype(data[col].dtype,np.number):
      vals = sorted(data[col].unique())
      splits = [(vals[i]+vals[i+1])/2 for i in range(0,len(vals)-1) ]
    else:
      splits = [None]

    for s in splits:
      if s is not None:
        subsets = [ data[data[col]<=s ],data[data[col]>s]]
      else:
        subsets =  [data[data[col]==v ] for v in data[col].unique()]
      weighted = get_weighted_score(subsets,target,method)

      if method == "cart":
        current_score = weighted
        is_better = ( best_score is None or current_score < best_score)
      else:
        gain = entropy(data[target]) - weighted
        if method == "c45":
          si = -sum((len(sub)/len(data))*np.log2(len(sub)/len(data)+1e-9) for sub in subsets )
          current_score = gain/(si + 1e-9)
        else:
          current_score = gain
        is_better = ( best_score is None or best_score > current_score)
      if is_better:
        best_score = current_score
        best_feat,best_val = col,s
  return best_feat,best_val

In [ ]:
def build_tree(data, method="id3", target="Play"):
    # Base Cases
    if len(data[target].unique()) == 1: return data[target].iloc[0]
    if len(data.columns) == 1: return data[target].mode()[0]

    feat, val = find_best_split(data, method, target)
    if feat is None: return data[target].mode()[0]

    tree = {}
    if val is not None: # Numerical
        tree[f"{feat} <= {val:.2f}"] = {
            "True": build_tree(data[data[feat] <= val].drop(columns=feat), method, target),
            "False": build_tree(data[data[feat] > val].drop(columns=feat), method, target)
        }
    else: # Categorical
        tree[feat] = {v: build_tree(data[data[feat] == v].drop(columns=feat), method, target)
                      for v in data[feat].unique()}
    return tree

In [ ]:
cart_tree = build_tree(data,"cart")
print(cart_tree)

{'Humidity': {'High': 'Yes', 'Normal': {'Temperature': {'Hot': 'Yes', 'Cool': {'Outlook': {'Rain': {'Age <= 37.00': {'True': 'Yes', 'False': {'Wind': {'Strong': 'No'}}}}, 'Overcast': 'No'}}, 'Mild': {'Outlook': {'Overcast': 'No', 'Sunny': 'Yes'}}}}}}


In [ ]:
id3_tree = build_tree(data,"id3")
print(cart_tree)

{'Humidity': {'High': 'Yes', 'Normal': {'Temperature': {'Hot': 'Yes', 'Cool': {'Outlook': {'Rain': {'Age <= 37.00': {'True': 'Yes', 'False': {'Wind': {'Strong': 'No'}}}}, 'Overcast': 'No'}}, 'Mild': {'Outlook': {'Overcast': 'No', 'Sunny': 'Yes'}}}}}}


In [ ]:
c45_tree = build_tree(data,"c45")
print(c45_tree)

{'Age <= 36.50': {'True': {'Temperature': {'Hot': 'Yes', 'Cool': {'Outlook': {'Overcast': {'Humidity': {'High': 'Yes', 'Normal': 'No'}}, 'Rain': 'Yes'}}, 'Mild': {'Wind': {'Strong': {'Outlook': {'Overcast': 'No', 'Sunny': 'Yes'}}}}}}, 'False': {'Outlook': {'Overcast': {'Wind': {'Strong': {'Temperature': {'Hot': 'Yes', 'Mild': 'Yes', 'Cool': 'No'}}, 'Weak': 'Yes'}}, 'Sunny': 'Yes', 'Rain': {'Wind': {'Strong': {'Temperature': {'Cool': {'Humidity': {'Normal': 'No', 'High': 'Yes'}}, 'Hot': 'Yes'}}}}}}}}
